# Mapping drugs: preparing the Boiler input files

This notebook shows the procedure for converting source drug codes into the input files required by the OMOP Drug Boiler (the `drug_concept_stage`, `internal_relationship_stage`, `ds_stage`, and `relationship_to_concept` files).

There is currently no gold standard for drugs, so this notebook does not include any evaluation.

The procedure has three stages:

1. **Structuring**: an LLM splits each source drug into its ingredients, dose forms, strengths, brand names, suppliers, etc., and separates drugs from devices.
2. **Normalization**: the structured data is converted into the Boiler staging tables.
3. **Mapping**: the staged drug attributes are mapped to standard OMOP concepts, producing `relationship_to_concept`.

![Notebook diagram](../assets/AriadneDiagramDrugs.png)

## Setup

Before running this notebook, make sure the environment is set up as described in the README, including the credentials for the LLM API and the connection to the OHDSI Vocabularies database. Settings for the drug pipeline are read by `ConfigDrugMapping`.

All results are stored in the `data/notebook_results/drug_results` folder. Most code blocks load results from file if they already exist, to save time and costs. **Delete those files to rerun the corresponding steps.**

In [1]:
from pathlib import Path
import logging
import pandas as pd
from dotenv import load_dotenv

from ariadne.utils.config_drug_mapping import ConfigDrugMapping

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")

project_root = Path.cwd().parent.parent
load_dotenv()

config_drug_mapping = ConfigDrugMapping()

results_folder = project_root / "data" / "notebook_results" / "drug_results"
results_folder.mkdir(parents=True, exist_ok=True)

## Source data

The input is a CSV file with one row per source drug, plus a markdown file describing the columns of that CSV (the data dictionary), which is given to the LLM as context.

Set the paths and the names of the columns holding the unique drug code and the full drug name below. Here we use the sample data in the `data/sample_data` folder.

In [2]:
input_csv = project_root / "data" / "sample_data" / "drug_codes_2_sample.csv"
input_dictionary_md = project_root / "data" / "sample_data" / "drug_codes_2_dictionary.md"
drug_code_column = "APPID"
full_drug_name_column = "NM_AMPP"

source_df = pd.read_csv(input_csv, dtype=str)
source_df.head()

,__record_tag_AMPP,APID,APPID,COMBPACKCD,DISCCD,DISCDT,INVALID_AMPP,LEGAL_CATCD,NM_AMPP,SUBP,...,__record_tag_VTM,ABBREVNM_VTM,INVALID_VTM,NM_VTM,VTMIDDT,VTMIDPREV,AGGREGATED_INGREDIENTS,AGGREGATED_ROUTES,DOSE_FORM,SUPPLIER_NAME
0,AMPP,37515811000001101,37515911000001106,NaN,NaN,NaN,NaN,3,Latuda 18.5mg tablets (CST Pharma Ltd) 28 tablet,NaN,...,VTM,NaN,NaN,Lurasidone,2023-02-06,2.66643110000011e+16,Lurasidone hydrochloride (703116009) 18.5 nan,Oral,Oral tablet,CST Pharma Ltd
1,AMPP,35547611000001102,35547811000001103,NaN,NaN,NaN,NaN,3,Sertraline 75mg/5ml oral suspension (Special O...,NaN,...,VTM,NaN,NaN,Sertraline,2023-02-06,321958004.0,Sertraline hydrochloride (96211006) 15.0 nan,Oral,Oral suspension,Special Order
2,AMPP,29643811000001108,29643911000001103,NaN,1.0,2023-01-01,NaN,4,Juzo Hostess class 2 (23-32mmHg) tights open t...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Juzo UK Ltd
3,AMPP,19640011000001108,19640111000001109,NaN,NaN,NaN,NaN,4,Sigvaris Advance 1 low compression (14-18mmHg)...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sigvaris Britain Ltd
4,AMPP,24789511000001108,24789611000001107,NaN,NaN,NaN,NaN,4,Sigvaris Essential Comfortable class 2 (23-32m...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sigvaris Britain Ltd


## Stage 1: Structuring the drugs

The `LlmDrugStructurer` uses the LLM, guided by the settings in `drug_structuring` and the data dictionary, to extract structured information from each source drug. It returns:

 - `classification_df`: whether each source code is a drug or a device
 - `ingredient_df`: the ingredients (and their strengths) per drug
 - `drug_df`: drug-level attributes such as dose form, brand name, and supplier
 - `device_df`: the source codes classified as devices

We also write a debug version of the ingredient file that includes the full drug name, to make manual review easier.

In [3]:
from types import SimpleNamespace
from ariadne.llm_mapping.llm_drug_structurer import LlmDrugStructurer

classify_path = results_folder / "classification.csv"
ingredient_path = results_folder / "ingredients.csv"
ingredient_debug_path = results_folder / "ingredients_with_full_drug_name.csv"
drug_path = results_folder / "drugs.csv"
device_path = results_folder / "devices.csv"

if all(p.exists() for p in [classify_path, ingredient_path, ingredient_debug_path, drug_path, device_path]):
    structured = SimpleNamespace(
        classification_df=pd.read_csv(classify_path, dtype=str),
        ingredient_df=pd.read_csv(ingredient_path, dtype=str),
        drug_df=pd.read_csv(drug_path, dtype=str),
        device_df=pd.read_csv(device_path, dtype=str),
    )
    print("Loaded structured drugs from file.")
else:
    structurer = LlmDrugStructurer(
        settings=config_drug_mapping.drug_structuring,
        dictionary_markdown_file=input_dictionary_md,
    )
    structured = structurer.structure_drugs(source_df, drug_code_column=drug_code_column)
    structured.classification_df.to_csv(classify_path, index=False)
    structured.ingredient_df.to_csv(ingredient_path, index=False)
    structured.drug_df.to_csv(drug_path, index=False)
    structured.device_df.to_csv(device_path, index=False)

    ingredient_debug_df = structured.ingredient_df.copy()
    if full_drug_name_column not in ingredient_debug_df.columns and full_drug_name_column in source_df.columns:
        ingredient_debug_df = ingredient_debug_df.merge(
            source_df[[drug_code_column, full_drug_name_column]].drop_duplicates(subset=[drug_code_column]),
            left_on="drug_code",
            right_on=drug_code_column,
            how="left",
        )
    ingredient_debug_df.to_csv(ingredient_debug_path, index=False)

structured.ingredient_df.head()

2026-11-13 00:02:30,922  Starting drug structuring for rows_total=250 batch_size=25
2026-11-13 00:02:30,957  Drug structuring progress rows_processed=25/250 live_cost_usd=0.000000 cache_hits=4 api_calls=0
2026-11-13 00:02:30,987  Drug structuring progress rows_processed=50/250 live_cost_usd=0.000000 cache_hits=8 api_calls=0
2026-11-13 00:02:31,017  Drug structuring progress rows_processed=75/250 live_cost_usd=0.000000 cache_hits=12 api_calls=0
2026-11-13 00:02:31,047  Drug structuring progress rows_processed=100/250 live_cost_usd=0.000000 cache_hits=16 api_calls=0
2026-11-13 00:02:31,079  Drug structuring progress rows_processed=125/250 live_cost_usd=0.000000 cache_hits=20 api_calls=0
2026-11-13 00:02:31,108  Drug structuring progress rows_processed=150/250 live_cost_usd=0.000000 cache_hits=24 api_calls=0
2026-11-13 00:02:31,139  Drug structuring progress rows_processed=175/250 live_cost_usd=0.000000 cache_hits=28 api_calls=0
2026-11-13 00:02:31,168  Drug structuring progress rows_proc

,drug_code,ingredient_name,ingredient_code,amount_value,amount_unit,numerator_value,numerator_unit,denominator_value,denominator_unit
0,37515911000001106,Lurasidone,776588007,18.5,mg,NaN,None,NaN,None
1,35547811000001103,Sertraline,777521008,NaN,None,75.0,mg,5.0,mL
2,19634311000001104,Letrozole,776502006,2.5,mg,NaN,None,NaN,None
3,34025911000001108,Oxycodone,777027001,NaN,None,20.0,mg,2.0,mL
4,21381011000001107,Glycopyrronium bromide,13298001000001104,NaN,None,0.5,mg,NaN,mL


In [4]:
structured.drug_df.head()

,drug_code,full_product_name,brand_name,brand_code,supplier_name,supplier_code,dose_form,box_size
0,37515911000001106,Latuda 18.5 mg oral tablet 28 tablets,Latuda,None,CST Pharma,24059811000001102,oral tablet,28.0
1,35547811000001103,Sertraline 75 mg/5 ml oral suspension 1 ml,None,None,None,None,oral suspension,NaN
2,19634311000001104,Letrozole 2.5 mg oral tablet 14 tablets,None,None,Dexcel-Pharma,2911401000001108,oral tablet,14.0
3,34025911000001108,Shortec 20 mg/2 ml solution for injection ampo...,Shortec,None,Qdem Pharmaceuticals,20921311000001103,solution for injection,5.0
4,21381011000001107,Glycopyrronium bromide 0.05% solution 600 ml,None,None,None,None,None,NaN


## Stage 2: Normalizing the drug attributes

`normalize_structured_drugs` converts the structured output into the Boiler staging tables:

 - `drug_concept_stage`: all source drugs and their attributes (ingredients, dose forms, brand names, suppliers, units)
 - `internal_relationship_stage`: relationships between drugs and their attributes
 - `ds_stage`: drug strength information

In [5]:
from ariadne.llm_mapping.llm_drug_structurer import normalize_structured_drugs

drug_concept_stage_path = results_folder / "drug_concept_stage.csv"
internal_relationship_stage_path = results_folder / "internal_relationship_stage.csv"
ds_stage_path = results_folder / "ds_stage.csv"

if all(p.exists() for p in [drug_concept_stage_path, internal_relationship_stage_path, ds_stage_path]):
    normalized = SimpleNamespace(
        drug_concept_stage=pd.read_csv(drug_concept_stage_path, dtype=str),
        internal_relationship_stage=pd.read_csv(internal_relationship_stage_path, dtype=str),
        ds_stage=pd.read_csv(ds_stage_path, dtype=str),
    )
    print("Loaded normalized drugs from file.")
else:
    normalized = normalize_structured_drugs(structured)
    normalized.drug_concept_stage.to_csv(drug_concept_stage_path, index=False)
    normalized.internal_relationship_stage.to_csv(internal_relationship_stage_path, index=False)
    normalized.ds_stage.to_csv(ds_stage_path, index=False)

normalized.drug_concept_stage.head()

,concept_name,domain_id,concept_class_id,concept_code
0,g,Unit,Unit,g
1,h,Unit,Unit,h
2,mL,Unit,Unit,mL
3,mg,Unit,Unit,mg
4,patch,Unit,Unit,patch


In [6]:
normalized.ds_stage.head()

,drug_concept_code,ingredient_concept_code,amount_value,amount_unit,numerator_value,numerator_unit,denominator_value,denominator_unit,box_size
0,37515911000001106,776588007,18.5,mg,NaN,None,NaN,None,28.0
1,35547811000001103,777521008,NaN,None,75.0,mg,5.0,mL,NaN
2,19634311000001104,776502006,2.5,mg,NaN,None,NaN,None,14.0
3,34025911000001108,777027001,NaN,None,20.0,mg,2.0,mL,5.0
4,21381011000001107,13298001000001104,NaN,None,0.5,mg,NaN,mL,NaN


## Stage 3: Mapping to standard concepts

Finally, the `DrugMapper` maps the attributes in `drug_concept_stage` (ingredients, dose forms, brand names, suppliers, units) to standard OMOP concepts. The result is the `relationship_to_concept` file.

In [7]:
from ariadne.llm_mapping.drug_mapper import DrugMapper

relationship_to_concept_path = results_folder / "relationship_to_concept.csv"

if relationship_to_concept_path.exists():
    relationship_to_concept = pd.read_csv(relationship_to_concept_path, dtype=str)
    print("Loaded relationship_to_concept from file.")
else:
    drug_mapper = DrugMapper(config_drug_mapping)
    relationship_to_concept = drug_mapper.map_drug_concepts(normalized.drug_concept_stage)
    relationship_to_concept.to_csv(relationship_to_concept_path, index=False)

relationship_to_concept.head(10)

Parquet files already exist in folder E:\git\Ariadne\data\terms_drug_ingredient. Skipping download.
spaCy model 'en_core_web_sm' loaded successfully.
Index loaded from E:\git\Ariadne\data\verbatim_drug_mapping_index_ingredient.pkl
Processing term 'Glycopyrronium bromide'
Processing term 'Noradrenaline'
Processing term 'Fenofibrate micronised'
Processing term 'indapamide hemihydrate'
Processing term 'paracetamol'
Processing term 'Neostigmine methylsulfate'
Processing term 'Betamethasone valerate'
Processing term 'Valaciclovir'
Processing term 'Apple mistletoe'
Processing term 'Arachis oil extract of crude coal tar'
Processing term 'Cade oil'
Processing term 'Coal tar solution'
Processing term 'Clobetasol propionate'
Processing term 'Etidronate disodium'
Processing term 'Hepar sulfuris'
Parquet files already exist in folder E:\git\Ariadne\data\terms_drug_ingredient. Skipping download.
spaCy model 'en_core_web_sm' loaded successfully.
Index loaded from E:\git\Ariadne\data\verbatim_drug_ma

,concept_code,source_name,mapped_concept_id,mapped_concept_name,drug_class_id
0,776588007,Lurasidone,40230761.0,lurasidone,Ingredient
1,777521008,Sertraline,739138.0,sertraline,Ingredient
2,776502006,Letrozole,1315946.0,letrozole,Ingredient
3,777027001,Oxycodone,1124957.0,oxycodone,Ingredient
4,777722004,Testosterone,1636780.0,testosterone,Ingredient
5,775543001,Diamorphine,19022417.0,heroin,Ingredient
6,776713006,Metformin,1503297.0,metformin,Ingredient
7,777294006,Pramipexole,720810.0,pramipexole,Ingredient
8,777388000,Rabeprazole,911735.0,rabeprazole,Ingredient
9,777905007,Ustekinumab,40161532.0,ustekinumab,Ingredient


## Boiler input files

The following files in the results folder are the inputs for the Drug Boiler:

In [8]:
for name in ["drug_concept_stage", "internal_relationship_stage", "ds_stage", "relationship_to_concept"]:
    path = results_folder / f"{name}.csv"
    print(f"{path}: {'found' if path.exists() else 'MISSING'}")

E:\git\Ariadne\data\notebook_results\drug_results\drug_concept_stage.csv: found
E:\git\Ariadne\data\notebook_results\drug_results\internal_relationship_stage.csv: found
E:\git\Ariadne\data\notebook_results\drug_results\ds_stage.csv: found
E:\git\Ariadne\data\notebook_results\drug_results\relationship_to_concept.csv: found
